# XDS zone-axis deviation table

This notebook parses XDS geometry and reports, for every frame, the deviation angle between the beam and the cubic families <001> and <011>.


In [60]:
from pathlib import Path
import numpy as np
import pandas as pd

from cred_sim.xds import load_xds_geometry
from cred_sim.geometry import rotation_matrix, spindle_angle_deg


In [61]:
# Point this at your XDS folder
xds_dir = Path("/home/bubl3932/files/3DED-DATA/LTA/LTA1/xds")  # change to your real XDS folder

geom = load_xds_geometry(xds_dir, prefer_ascii_header=True)

print("Loaded geometry:")
print(f"  start={geom.starting_frame}, angle={geom.starting_angle_deg:.3f}, osc={geom.oscillation_range_deg:.3f}")
print(f"  beam_dir={geom.beam_dir_unit()}")


Loaded geometry:
  start=1, angle=-45.000, osc=0.120
  beam_dir=[0. 0. 1.]


In [62]:
# Build <001> and <011> families (unique up to sign)
zone_001 = [(1, 0, 0), (0, 1, 0), (0, 0, 1)]
zone_011 = [(0, 1, 1), (0, 1, -1), (0, -1, 1), (0, -1, -1),
            (1, 0, 1), (1, 0, -1), (-1, 0, 1), (-1, 0, -1),
            (1, 1, 0), (1, -1, 0), (-1, 1, 0), (-1, -1, 0)]

def frame_rotation(geom, frame_idx, orientation_reference="phi0"):
    axis = geom.rotation_axis_unit()
    if orientation_reference == "phi0":
        phi_deg = spindle_angle_deg(geom.starting_angle_deg, geom.oscillation_range_deg, geom.starting_frame, frame_idx)
    elif orientation_reference == "frame0":
        phi_deg = geom.oscillation_range_deg * (frame_idx - geom.starting_frame)
    else:
        raise ValueError("orientation_reference must be 'phi0' or 'frame0'")
    return rotation_matrix(axis, np.deg2rad(phi_deg))

def angle_to_beam(geom, frame_idx, uvw, orientation_reference="phi0"):
    A = geom.direct_basis_A()
    beam = geom.beam_dir_unit()
    R = frame_rotation(geom, frame_idx, orientation_reference)
    vec = A @ np.array(uvw, dtype=float)
    vec = vec / np.linalg.norm(vec)
    vec_lab = R @ vec
    cosang = np.clip(abs(np.dot(vec_lab, beam)), -1.0, 1.0)
    return float(np.degrees(np.arccos(cosang)))

def best_angle_for_family(geom, frame_idx, family, orientation_reference="phi0"):
    best = (1e9, None)
    for uvw in family:
        ang = angle_to_beam(geom, frame_idx, uvw, orientation_reference)
        if ang < best[0]:
            best = (ang, uvw)
    return best


In [ ]:
# Per-frame orientation matrices (crystal -> lab) and beam in crystal frame
def orthonormalize_columns(A):
    Q, _ = np.linalg.qr(A)
    if np.linalg.det(Q) < 0:
        Q[:, 2] *= -1
    return Q

def orientation_matrix_crystal_to_lab(geom, frame_idx, orientation_reference="phi0", lab_offset=None):
    A = geom.direct_basis_A()
    U0 = orthonormalize_columns(A)
    R = frame_rotation(geom, frame_idx, orientation_reference)
    U = R @ U0
    if lab_offset is not None:
        U = lab_offset @ U
    return U

# Pick the reference convention used for scan angles
orientation_reference = "phi0"  # change to "frame0" if needed

# Optional global lab-frame offset (tweak if you suspect a constant rotation)
lab_offset_axis = np.array([0.0, 0.0, 1.0])
lab_offset_deg = 0.0
lab_offset = rotation_matrix(lab_offset_axis, np.deg2rad(lab_offset_deg))

if geom.data_range is None:
    frame_start, frame_end = geom.starting_frame, geom.starting_frame
else:
    frame_start, frame_end = geom.data_range

rows = []
for frame_idx in range(frame_start, frame_end + 1):
    U = orientation_matrix_crystal_to_lab(geom, frame_idx, orientation_reference, lab_offset)
    beam_lab = geom.beam_dir_unit()
    beam_crys = U.T @ beam_lab
    rows.append({
        "frame": frame_idx,
        "beam_crys_x": float(beam_crys[0]),
        "beam_crys_y": float(beam_crys[1]),
        "beam_crys_z": float(beam_crys[2]),
        "U11": float(U[0, 0]), "U12": float(U[0, 1]), "U13": float(U[0, 2]),
        "U21": float(U[1, 0]), "U22": float(U[1, 1]), "U23": float(U[1, 2]),
        "U31": float(U[2, 0]), "U32": float(U[2, 1]), "U33": float(U[2, 2]),
    })

df_orient = pd.DataFrame(rows)
df_orient.head(10)


In [65]:
# Compute per-frame deviation table
orientation_reference = "phi0"  # change to "frame0" if needed

if geom.data_range is None:
    frame_start, frame_end = geom.starting_frame, geom.starting_frame
else:
    frame_start, frame_end = geom.data_range

rows = []
for frame_idx in range(frame_start, frame_end + 1):
    ang001, uvw001 = best_angle_for_family(geom, frame_idx, zone_001, orientation_reference)
    ang011, uvw011 = best_angle_for_family(geom, frame_idx, zone_011, orientation_reference)
    rows.append({
        "frame": frame_idx,
        "ang_001_deg": ang001,
        "best_001_uvw": uvw001,
        "ang_011_deg": ang011,
        "best_011_uvw": uvw011,
    })

df = pd.DataFrame(rows)
df.head(60)


,frame,ang_001_deg,best_001_uvw,ang_011_deg,best_011_uvw
0,1,18.351833,"(1, 0, 0)",29.495431,"(1, 1, 0)"
1,2,18.356263,"(1, 0, 0)",29.569329,"(1, 1, 0)"
2,3,18.361449,"(1, 0, 0)",29.643502,"(1, 1, 0)"
3,4,18.367390,"(1, 0, 0)",29.717947,"(1, 1, 0)"
4,5,18.374085,"(1, 0, 0)",29.792662,"(1, 1, 0)"
5,6,18.381534,"(1, 0, 0)",29.867646,"(1, 1, 0)"
6,7,18.389736,"(1, 0, 0)",29.942897,"(1, 1, 0)"
7,8,18.398690,"(1, 0, 0)",30.018412,"(1, 1, 0)"
8,9,18.408394,"(1, 0, 0)",30.094189,"(1, 1, 0)"
9,10,18.418848,"(1, 0, 0)",30.170227,"(1, 1, 0)"


In [ ]:
# Inspect specific frames and show small per-frame changes
pd.set_option("display.precision", 6)
frames_check = [1, 2, 3, 4, 5, 50, 51, 592, 593]
df[df["frame"].isin(frames_check)]
